In [1]:
import os
import pickle
import numpy as np
import pandas as pd
import neurokit2 as nk
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.signal import butter, filtfilt, find_peaks

from sklearn.model_selection import LeaveOneGroupOut
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier

from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score, f1_score, confusion_matrix,
    matthews_corrcoef, roc_auc_score
)

In [7]:
DATA_PATH = "../data/WESAD"
FS = 700

SUBJECT_IDS = (
    [f"S{i}" for i in range(2, 12)] +
    [f"S{i}" for i in range(13, 18)]
)

WINDOW_SEC = 120
STEP_SEC = 60
SEQ_LEN = 5

def load_subject(path):
    with open(path, "rb") as f:
        return pickle.load(f, encoding="latin1")


In [4]:
# ===================== EDA FEATURES =====================
def tonic_phasic(eda, fs):
    eda = eda[~np.isnan(eda)]
    b, a = butter(4, 0.05 / (fs / 2), btype="low")
    tonic = filtfilt(b, a, eda)
    phasic = eda - tonic
    return tonic, phasic

def extract_eda_features(eda, fs):
    tonic, phasic = tonic_phasic(eda, fs)
    peaks, _ = find_peaks(phasic, height=0.01, distance=fs)
    return len(peaks), np.trapz(np.abs(phasic)) / fs, np.mean(tonic)

def build_eda_table(subject):
    eda = subject["signal"]["chest"]["EDA"]
    labels = subject["label"]

    WINDOW = FS * WINDOW_SEC
    STEP = FS * STEP_SEC
    rows = []

    for i in range(0, len(eda) - WINDOW, STEP):
        win_eda = eda[i:i + WINDOW]
        win_labels = labels[i:i + WINDOW]

        if np.isnan(win_eda).any():
            continue

        majority = np.bincount(win_labels).argmax()
        if majority not in [1, 2, 3, 4]:
            continue

        label_bin = 1 if majority == 2 else 0
        scr, auc, tonic = extract_eda_features(win_eda, FS)

        rows.append({
            "Time": i / FS,
            "Label": label_bin,
            "EDA_SCR_count": scr,
            "EDA_Phasic_AUC": auc,
            "EDA_Tonic_Mean": tonic
        })

    return pd.DataFrame(rows)

In [5]:
def build_hrv_table(subject):
    ecg = subject["signal"]["chest"]["ECG"]
    labels = subject["label"]

    cleaned = nk.ecg_clean(ecg, sampling_rate=FS)
    _, rpeaks = nk.ecg_peaks(cleaned, sampling_rate=FS)
    rpeaks_idx = rpeaks["ECG_R_Peaks"]

    WINDOW = FS * WINDOW_SEC
    STEP = FS * STEP_SEC
    rows = []

    for start in range(0, len(ecg) - WINDOW, STEP):
        end = start + WINDOW
        peaks = rpeaks_idx[(rpeaks_idx >= start) & (rpeaks_idx < end)] - start
        if len(peaks) <= 2:
            continue

        rr_ms = np.diff(peaks) / FS * 1000
        if len(rr_ms) < 2:
            continue

        try:
            rpeaks_clean = nk.intervals_to_peaks(rr_ms, sampling_rate=FS)
            hrv_t = nk.hrv_time(rpeaks_clean, sampling_rate=FS, show=False)
            hrv_f = nk.hrv_frequency(rpeaks_clean, sampling_rate=FS, show=False)

            label_bin = np.bincount((labels[start:end] == 2).astype(int)).argmax()

            rows.append({
                "Time": (start + end) / 2 / FS,
                "Label": label_bin,
                "HRV_RMSSD": hrv_t["HRV_RMSSD"].values[0],
                "HRV_SDNN": hrv_t["HRV_SDNN"].values[0],
                "HRV_MeanNN": hrv_t["HRV_MeanNN"].values[0],
                "HRV_LF": hrv_f["HRV_LF"].values[0],
                "HRV_HF": hrv_f["HRV_HF"].values[0],
                "HRV_LFHF": hrv_f["HRV_LFHF"].values[0],
            })
        except Exception:
            continue

    return pd.DataFrame(rows)


In [10]:
# ===================== FEATURE EXTRACTION =====================
eda_all, hrv_all = [], []

for sid in SUBJECT_IDS:
    subject = load_subject(f"{DATA_PATH}/{sid}/{sid}.pkl")

    eda_df = build_eda_table(subject)
    hrv_df = build_hrv_table(subject)

    eda_df["Subject"] = sid
    hrv_df["Subject"] = sid

    eda_all.append(eda_df)
    hrv_all.append(hrv_df)

EDA_FEATURES = pd.concat(eda_all, ignore_index=True)
HRV_FEATURES = pd.concat(hrv_all, ignore_index=True)

merged_df = pd.merge(
    EDA_FEATURES,
    HRV_FEATURES,
    on=["Time", "Label", "Subject"],
    how="inner"
)

print("\n===== DATASET SUMMARY =====")
print("Total samples:", len(merged_df))
print("Total subjects:", merged_df["Subject"].nunique())
print("Subjects:", sorted(merged_df["Subject"].unique()))

print("\nLabel distribution:")
print(merged_df["Label"].value_counts())

print("\nWindows per subject:")
print(merged_df.groupby("Subject").size())

# ===================== TABULAR DATA =====================
X = merged_df.drop(columns=["Time", "Label", "Subject"])
y = merged_df["Label"].values
groups = merged_df["Subject"].values

logo = LeaveOneGroupOut()

print("\n===== FEATURE STATISTICS (mean / std) =====")
print(X.describe().T[["mean", "std"]])

/tmp/ipykernel_18734/2276676472.py:12: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return len(peaks), np.trapz(np.abs(phasic)) / fs, np.mean(tonic)
/tmp/ipykernel_18734/2276676472.py:12: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return len(peaks), np.trapz(np.abs(phasic)) / fs, np.mean(tonic)
/tmp/ipykernel_18734/2276676472.py:12: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return len(peaks), np.trapz(np.abs(phasic)) / fs, np.mean(tonic)
/tmp/ipykernel_18734/2276676472.py:12: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return len(peaks), np.trapz(np.abs(phasic)) / fs, np.mean(tonic)
/tmp/ipykernel_18734/2276676472.


===== DATASET SUMMARY =====
Total samples: 744
Total subjects: 15
Subjects: ['S10', 'S11', 'S13', 'S14', 'S15', 'S16', 'S17', 'S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9']

Label distribution:
Label
0    592
1    152
Name: count, dtype: int64

Windows per subject:
Subject
S10    51
S11    50
S13    50
S14    50
S15    51
S16    50
S17    50
S2     47
S3     50
S4     49
S5     48
S6     48
S7     51
S8     51
S9     48
dtype: int64

===== FEATURE STATISTICS (mean / std) =====
                      mean         std
EDA_SCR_count    90.250000    3.161746
EDA_Phasic_AUC   15.532891   11.991795
EDA_Tonic_Mean    4.472362    3.270755
HRV_RMSSD        56.921841   35.417955
HRV_SDNN         81.784960   35.007956
HRV_MeanNN      806.377773  146.778378
HRV_LF            0.033005    0.017154
HRV_HF            0.016540    0.017453
HRV_LFHF          4.301113    5.691308


In [19]:
sid = "S2"
subject = load_subject(f"{DATA_PATH}/{sid}/{sid}.pkl")
print(subject["signal"]['wrist'].keys())

dict_keys(['ACC', 'BVP', 'EDA', 'TEMP'])
